# ULMFiT 模型训练与导出

本 notebook 用于训练 AG News 文本分类模型（ULMFiT / AWD-LSTM），并导出为 `.pkl` 文件，供微服务部署使用。

**运行环境**：Google Colab（GPU 运行时 T4）

**预计耗时**：约 5-6 分钟

In [ ]:
!pip -q install fastai datasets

In [ ]:
import numpy as np
import pandas as pd
import random
import time

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from fastai.text.all import *

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

## 1. 加载数据集并采样

In [ ]:
ds = load_dataset("ag_news")

train_df = ds["train"].to_pandas()
test_df  = ds["test"].to_pandas()

SAMPLE_N = 20000
train_df_sample = train_df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)

train_df2, valid_df2 = train_test_split(
    train_df_sample,
    test_size=0.2,
    random_state=42,
    stratify=train_df_sample["label"]
)

train_df2 = train_df2.reset_index(drop=True)
valid_df2 = valid_df2.reset_index(drop=True)

print(f"Train: {len(train_df2)}, Valid: {len(valid_df2)}, Test: {len(test_df)}")

In [ ]:
label_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

train_df2 = train_df2.copy()
valid_df2 = valid_df2.copy()
test_df = test_df.copy()

train_df2["label_str"] = train_df2["label"].map(label_names)
valid_df2["label_str"] = valid_df2["label"].map(label_names)
test_df["label_str"]   = test_df["label"].map(label_names)

print(train_df2[["text", "label_str"]].head())

## 2. 微调语言模型

In [ ]:
lm_df = pd.concat([train_df2[["text"]], valid_df2[["text"]]], axis=0).reset_index(drop=True)

dls_lm = TextDataLoaders.from_df(
    lm_df,
    text_col="text",
    is_lm=True,
    valid_pct=0.1,
    seed=42,
    bs=64
)

learn_lm = language_model_learner(
    dls_lm,
    AWD_LSTM,
    pretrained=True,
    drop_mult=0.5,
    metrics=[Perplexity()]
)

start_lm = time.time()
learn_lm.fine_tune(1, 1e-2)
lm_time = time.time() - start_lm

print(f"\nLM fine-tune time: {lm_time:.2f}s")

learn_lm.save_encoder("agnews_enc")

## 3. 训练文本分类器

In [ ]:
dls_clas = TextDataLoaders.from_df(
    train_df2,
    valid_df=valid_df2,
    text_col="text",
    label_col="label_str",
    text_vocab=dls_lm.vocab,
    bs=32,
    seed=42
)

learn_clas = text_classifier_learner(
    dls_clas,
    AWD_LSTM,
    pretrained=True,
    drop_mult=0.5,
    metrics=accuracy
)

learn_clas.load_encoder("agnews_enc")

start_clf = time.time()
learn_clas.fine_tune(3, 2e-2)
clf_time = time.time() - start_clf

print(f"\nClassifier training time: {clf_time:.2f}s")
print(f"Total training time: {lm_time + clf_time:.2f}s")

## 4. 验证模型

In [ ]:
learn_clas.validate()

In [ ]:
test_dl = learn_clas.dls.test_dl(test_df["text"])
preds, _ = learn_clas.get_preds(dl=test_dl)
pred_idx = preds.argmax(dim=1).cpu().numpy()

label_vocab = learn_clas.dls.vocab[1]
pred_labels = [label_vocab[i] for i in pred_idx]

ulmfit_test_acc = (np.array(pred_labels) == test_df["label_str"].values).mean()
print(f"ULMFiT Test Accuracy: {ulmfit_test_acc:.4f}")

print(f"\nLabel vocab order: {list(label_vocab)}")

## 5. 导出模型

`learn.export()` 会把模型权重、tokenizer、vocab 全部打包，部署时只需 `load_learner()` 即可推理。

In [ ]:
learn_clas.export('ag_news_classifier.pkl')

import os
fsize = os.path.getsize('ag_news_classifier.pkl') / (1024 * 1024)
print(f"Model exported: ag_news_classifier.pkl ({fsize:.1f} MB)")

## 6. 快速验证导出的模型

In [ ]:
learn_loaded = load_learner('ag_news_classifier.pkl')

test_texts = [
    "NASA launches new Mars exploration mission with advanced rover technology",
    "Apple stock rises after strong quarterly earnings report",
    "Manchester United wins Champions League final",
    "UN Security Council holds emergency meeting on Middle East crisis"
]

for text in test_texts:
    pred_class, pred_idx, probs = learn_loaded.predict(text)
    print(f"Text: {text[:60]}...")
    print(f"  => {pred_class} (confidence: {probs.max():.4f})")
    print()

## 7. 下载模型文件

运行下面的 cell，浏览器将自动下载 `ag_news_classifier.pkl`。

下载后将文件放入项目的 `models/` 目录中。

In [ ]:
from google.colab import files
files.download('ag_news_classifier.pkl')